# Chapter 4 Study Lab: Data Treatment, Data Tests, Model Selection, and Development Tests

This notebook is designed to accompany Chapter 4, **First-Line Defense: Model Development / Model Owners**.

It follows the chapter's time-series / CCAR-style example and demonstrates:
- data collection and quality review;
- missing values, duplicates, outliers, and Winsorization;
- relevance and adequacy checks;
- skewness/kurtosis and transformations;
- stationarity and autocorrelation;
- correlation and VIF;
- candidate model selection using AIC/BIC and LASSO;
- calibration diagnostics;
- out-of-sample testing and benchmarking.

The dataset is synthetic and intentionally contains missing values, duplicate rows, outliers, and correlated predictors so that the tests produce meaningful findings.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import skew, kurtosis, mstats
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

DATA = Path("Ch004_CCARDemo_Data.csv")
if not DATA.exists():
    DATA = Path("/mnt/data/Ch004_CCARDemo_Data.csv")

df = pd.read_csv(DATA, parse_dates=["date"])
df.head()


## 1. Data integrity and completeness

Chapter 4 emphasizes missing-data checks, duplicate checks, data-type consistency, and documentation of any treatment decisions.


In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)

# Remove exact duplicates but keep an audit count.
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().sort_values("date").reset_index(drop=True)


## 2. Missing-data treatment

For demonstration, numerical missing values are imputed with the median. In production MRM work, the choice among deletion, median/mean imputation, regression-based methods, or other approaches should be justified based on the missingness mechanism, materiality, and intended model use.


In [ ]:
numeric_cols = [c for c in df.columns if c != "date"]
missing_before = df[numeric_cols].isna().sum()

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

print(pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": df[numeric_cols].isna().sum()
}))


## 3. Outlier detection and Winsorization

Chapter 4 discusses Z-scores, IQR rules, and Winsorization. Here we use the IQR rule for detection and 1%/99% Winsorization for treatment of the target variable.


In [ ]:
def iqr_outlier_mask(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)

mask = iqr_outlier_mask(df["fee_income"])
print("IQR outliers in fee_income:", int(mask.sum()))
print(df.loc[mask, ["date", "fee_income"]])

df["fee_income_raw"] = df["fee_income"]
lo, hi = df["fee_income"].quantile([0.01, 0.99])
df["fee_income"] = df["fee_income"].clip(lo, hi)

plt.figure(figsize=(9,4))
plt.plot(df["date"], df["fee_income_raw"], label="Raw")
plt.plot(df["date"], df["fee_income"], label="Winsorized")
plt.legend()
plt.title("Fee income: before and after outlier treatment")
plt.show()


## 4. Distribution and normalization tests

Skewness and kurtosis help identify variables whose distributions may benefit from transformation. Standardization can improve numerical stability and is essential for penalized methods such as LASSO.


In [ ]:
dist = []
for c in ["unemployment_rate","gdp_growth","inflation","treasury_10y","market_return","fee_income"]:
    dist.append([c, skew(df[c], bias=False), kurtosis(df[c], fisher=True, bias=False)])
pd.DataFrame(dist, columns=["variable","skewness","excess_kurtosis"])


## 5. Stationarity and autocorrelation

For a time-series forecasting model, Chapter 4 highlights the Augmented Dickey-Fuller test and autocorrelation analysis. The ADF null hypothesis is that the series has a unit root (nonstationary).


In [ ]:
def adf_summary(s):
    stat, pvalue, usedlag, nobs, crit, icbest = adfuller(s.dropna(), autolag="AIC")
    return {"ADF statistic": stat, "p-value": pvalue, "lags": usedlag, "nobs": nobs}

for c in ["fee_income","unemployment_rate","gdp_growth","inflation"]:
    print(c, adf_summary(df[c]))

from pandas.plotting import autocorrelation_plot
plt.figure(figsize=(8,4))
autocorrelation_plot(df["fee_income"])
plt.title("Autocorrelation: fee_income")
plt.show()


## 6. Correlation and multicollinearity (VIF)

The dataset intentionally includes `unemployment_alt`, which is highly correlated with `unemployment_rate`. This allows the VIF test to demonstrate how redundant predictors can destabilize coefficient estimates.


In [ ]:
predictors = ["unemployment_rate","unemployment_alt","gdp_growth","inflation","treasury_10y","market_return"]
corr = df[predictors].corr()
print(corr.round(3))

X_vif = sm.add_constant(df[predictors])
vif_table = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif_table


## 7. Time-ordered train/test split

For time-series applications, a chronological holdout is preferable to a random split because it better represents actual projection use.


In [ ]:
features_full = ["unemployment_rate","unemployment_alt","gdp_growth","inflation","treasury_10y","market_return"]
features_reduced = ["unemployment_rate","gdp_growth","inflation","treasury_10y","market_return"]

split = int(len(df)*0.80)
train, test = df.iloc[:split].copy(), df.iloc[split:].copy()

def fit_ols(data, features):
    X = sm.add_constant(data[features])
    return sm.OLS(data["fee_income"], X).fit()

full_model = fit_ols(train, features_full)
reduced_model = fit_ols(train, features_reduced)

print("Full model AIC/BIC:", full_model.aic, full_model.bic)
print("Reduced model AIC/BIC:", reduced_model.aic, reduced_model.bic)
print(reduced_model.summary())


## 8. LASSO variable selection

LASSO uses L1 regularization and can shrink unnecessary coefficients to zero. It is one of the variable-selection techniques cited in Chapter 4.


In [ ]:
X = train[features_full].values
y = train["fee_income"].values

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

tscv = TimeSeriesSplit(n_splits=5)
lasso = LassoCV(cv=tscv, random_state=42).fit(Xs, y)

coef = pd.Series(lasso.coef_, index=features_full)
print("Selected alpha:", lasso.alpha_)
print(coef.sort_values(key=np.abs, ascending=False))


## 9. Calibration diagnostics

Chapter 4's calibration checklist includes autocorrelation, heteroscedasticity, multicollinearity, residual diagnostics, and generalization.


In [ ]:
resid = reduced_model.resid

print("Durbin-Watson:", durbin_watson(resid))

bp = het_breuschpagan(resid, reduced_model.model.exog)
print("Breusch-Pagan LM p-value:", bp[1])
print("Breusch-Pagan F p-value:", bp[3])

white = het_white(resid, reduced_model.model.exog)
print("White test LM p-value:", white[1])

sm.qqplot(resid, line="45")
plt.title("Residual Q-Q plot")
plt.show()


## 10. Out-of-sample performance and benchmarking

The primary model is compared with a simple benchmark using only GDP growth and unemployment. This mirrors Chapter 4's emphasis on outcome analysis, benchmarking, and challenger models.


In [ ]:
def predict(model, data, features):
    X = sm.add_constant(data[features], has_constant="add")
    return model.predict(X)

pred_primary = predict(reduced_model, test, features_reduced)

benchmark_features = ["unemployment_rate","gdp_growth"]
benchmark_model = fit_ols(train, benchmark_features)
pred_benchmark = predict(benchmark_model, test, benchmark_features)

def metrics(y, pred):
    return {
        "RMSE": mean_squared_error(y, pred)**0.5,
        "R2": r2_score(y, pred)
    }

results = pd.DataFrame([
    {"model":"Primary reduced OLS", **metrics(test["fee_income"], pred_primary)},
    {"model":"Benchmark OLS", **metrics(test["fee_income"], pred_benchmark)}
])
results


In [ ]:
plt.figure(figsize=(9,4))
plt.plot(test["date"], test["fee_income"], marker="o", label="Actual")
plt.plot(test["date"], pred_primary, marker="o", label="Primary")
plt.plot(test["date"], pred_benchmark, marker="o", label="Benchmark")
plt.legend()
plt.title("Out-of-sample outcome analysis")
plt.show()


## 11. First-Line Development Conclusion Template

Document the conclusion rather than reporting test statistics without interpretation.

A good development conclusion should state:
1. whether the dataset is relevant, complete enough, and representative for the intended use;
2. what data defects were found and how they were treated;
3. whether key statistical assumptions are reasonably satisfied;
4. why the selected model is preferred to alternatives;
5. whether out-of-sample performance is acceptable;
6. what limitations and residual model risks remain;
7. what monitoring metrics and thresholds should be used after implementation.

This conclusion, together with code, data, testing evidence, limitations, and the monitoring plan, becomes part of the First-Line evidence package submitted for independent validation.
